# Landslide hazard step 01: intersections

Mirrors the coastal intersection workflow, with an added preprocessing step to reproject each landslide RP raster to Jamaica CRS (`EPSG:3448`) before running vector-raster intersections.


In [ ]:
import re
import subprocess
from pathlib import Path

import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')

landslide_rasters_root = base_path / 'dphil_paper_3/inputs/landslides/landslide_hazard_rp_maps'
networks_data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'
networks_metadata_csv = networks_data_root / 'networks/network_layers_hazard_intersections_details.csv'

output_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/landslide_hazard_network_intersections'
projected_rasters_path = output_path / 'projected_hazard_rasters_epsg3448'

vector_intersections_script = base_path / 'robyns_libraries/vector_raster_intersections.py'

jamaica_metric_grid_crs = 'EPSG:3448'

output_path.mkdir(parents=True, exist_ok=True)
projected_rasters_path.mkdir(parents=True, exist_ok=True)

print('Landslide rasters root:', landslide_rasters_root)
print('Networks metadata csv:', networks_metadata_csv)
print('Output path:', output_path)


In [ ]:
scenario_lookup = {
    'baseline_landslide_rps': 'baseline',
    'deforestation_landslide_rps': 'deforestation',
    'reforestation_landslide_rps': 'reforestation',
}

rp_pattern = re.compile(r'_(\d+)yr\.tif$', flags=re.IGNORECASE)

def reproject_raster_to_epsg3448(source_raster: Path, destination_raster: Path, destination_crs: str = 'EPSG:3448'):
    with rasterio.open(source_raster) as src:
        transform, width, height = calculate_default_transform(
            src.crs, destination_crs, src.width, src.height, *src.bounds
        )

        destination_profile = src.profile.copy()
        destination_profile.update(
            crs=destination_crs,
            transform=transform,
            width=width,
            height=height,
        )

        with rasterio.open(destination_raster, 'w', **destination_profile) as dst:
            for band_index in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band_index),
                    destination=rasterio.band(dst, band_index),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=destination_crs,
                    src_nodata=src.nodata,
                    dst_nodata=src.nodata,
                    resampling=Resampling.bilinear,
                )

hazard_rows = []

for tif_path in sorted(landslide_rasters_root.rglob('*.tif')):
    scenario_folder_name = tif_path.parent.name
    if scenario_folder_name not in scenario_lookup:
        continue

    scenario_name = scenario_lookup[scenario_folder_name]
    rp_match = rp_pattern.search(tif_path.name)
    if not rp_match:
        raise ValueError(f'Could not parse RP from filename: {tif_path.name}')

    return_period_years = int(rp_match.group(1))

    projected_filename = f'landslide_{scenario_name}_rp_{return_period_years}_epsg3448.tif'
    projected_raster = projected_rasters_path / projected_filename

    reproject_raster_to_epsg3448(tif_path, projected_raster, jamaica_metric_grid_crs)

    hazard_rows.append({
        'hazard': 'landslide',
        'scenario': scenario_name,
        'rp': return_period_years,
        'key': f'landslide_{scenario_name}_rp_{return_period_years}',
        'path': str(projected_raster),
        'fname': str(projected_raster),
        'source_path': str(tif_path),
    })

hazard_layers_table = pd.DataFrame(hazard_rows).sort_values(['scenario', 'rp']).reset_index(drop=True)

scenario_counts = hazard_layers_table.groupby('scenario').size().to_dict()
print('Hazard rasters found by scenario:', scenario_counts)
print('Total hazard rasters:', len(hazard_layers_table))

expected_scenarios = {'baseline', 'deforestation', 'reforestation'}
if set(scenario_counts.keys()) != expected_scenarios:
    raise ValueError(f'Expected scenarios {expected_scenarios} but found {set(scenario_counts.keys())}')

for scenario_name, count in scenario_counts.items():
    if count != 7:
        raise ValueError(f'Expected 7 rasters for {scenario_name}, found {count}')

hazard_layers_output_file = output_path / 'landslide_rasters_for_intersections.csv'
hazard_layers_table.to_csv(hazard_layers_output_file, index=False)

hazard_layers_table


In [ ]:
network_layers_table = pd.read_csv(networks_metadata_csv)
network_layers_table = network_layers_table[['path']].drop_duplicates().reset_index(drop=True)

# Same path fix used in the coastal workflow so paths resolve from common_incoming_data
network_layers_table['path'] = network_layers_table['path'].str.replace(
    r'^networks/',
    'networks/networks/',
    regex=True,
)

network_layers_output_file = output_path / 'network_layers_for_intersections.csv'
network_layers_table.to_csv(network_layers_output_file, index=False)

print('Network layers file:', network_layers_output_file)
print('Hazard layers file:', hazard_layers_output_file)
print('Network files (unique gpkg count):', len(network_layers_table))

network_layers_table


In [ ]:
run_intersections = True  # Set to True to run vector-raster intersections

if run_intersections:
    args = [
        'python',
        str(vector_intersections_script),
        str(network_layers_output_file),
        str(hazard_layers_output_file),
        str(output_path),
    ]
    print('* Start the processing of vector-raster intersections')
    print(args)
    subprocess.run(args, check=True)

print('* Done with the processing of vector-raster intersections')


## Completion Check

Run this after intersections to confirm all expected network outputs were produced.

In [ ]:
from pathlib import Path
import geopandas as gpd

intersections_output_dir = output_path

checks = {
    'roads': [
        'roads_splits__landslide_rasters_for_intersections__nodes.geoparquet',
        'roads_splits__landslide_rasters_for_intersections__edges.geoparquet',
    ],
    'rail': [
        'rail_splits__landslide_rasters_for_intersections__nodes.geoparquet',
        'rail_splits__landslide_rasters_for_intersections__edges.geoparquet',
    ],
    'ports': [
        'port_polygon_splits__landslide_rasters_for_intersections__areas.geoparquet',
    ],
    'airports': [
        'airport_polygon_splits__landslide_rasters_for_intersections__areas.geoparquet',
    ],
    'water_irrigation': [
        'irrigation_assets_NIC_splits__landslide_rasters_for_intersections__nodes.geoparquet',
        'irrigation_assets_NIC_splits__landslide_rasters_for_intersections__edges.geoparquet',
    ],
    'water_pipelines': [
        'pipelines_NWC_splits__landslide_rasters_for_intersections__edges.geoparquet',
    ],
    'water_potable': [
        'potable_facilities_NWC_splits__landslide_rasters_for_intersections__nodes.geoparquet',
    ],
    'water_wastewater': [
        'waste_water_facilities_NWC_splits__landslide_rasters_for_intersections__nodes.geoparquet',
    ],
    'energy': [
        'electricity_network_v3.1_splits__landslide_rasters_for_intersections__nodes.geoparquet',
        'electricity_network_v3.1_splits__landslide_rasters_for_intersections__edges.geoparquet',
    ],
    'buildings': [
        'buildings_assigned_economic_activity_splits__landslide_rasters_for_intersections__areas.geoparquet',
    ],
}

summary_rows = []
for group_name, filenames in checks.items():
    group_ok = True
    for filename in filenames:
        file_path = intersections_output_dir / filename
        exists = file_path.exists()
        row_count = None
        hazard_column_count = None

        if exists:
            gdf = gpd.read_parquet(file_path)
            row_count = len(gdf)
            hazard_columns = [
                c for c in gdf.columns if c.startswith('landslide_') and '_rp_' in c
            ]
            hazard_column_count = len(hazard_columns)
        else:
            group_ok = False

        summary_rows.append({
            'group': group_name,
            'file': filename,
            'exists': exists,
            'rows': row_count,
            'hazard_columns': hazard_column_count,
        })

    print(f"{group_name}: {'OK' if group_ok else 'MISSING'}")

summary_df = pd.DataFrame(summary_rows)
summary_df
